# 04 Error analysis

Where the model fails, and why. The full write up is `reports/error_analysis.md`,
regenerated by `make error`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, duckdb
pd.set_option('display.width', 160)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from src.features import build, NUMERIC, CATEGORICAL, TARGET
from src.model import preprocessor, SegmentUFRate
from src.error_analysis import calibration_table, by_group, value_band
from src.config import SEED
df = build(pd.read_parquet('../data/processed/projetos.parquet'))
s = df[df.in_model_sample]
tr, te = s[s.split=='train'], s[s.split=='test']
cols = NUMERIC + CATEGORICAL
m = Pipeline([('prep', preprocessor()),
              ('clf', LogisticRegression(max_iter=2000, random_state=SEED))])
m.fit(tr[cols], tr[TARGET].astype(int))
h = SegmentUFRate().fit(tr[cols], tr[TARGET].astype(int))
y = te[TARGET].astype(int).to_numpy()
p_model = m.predict_proba(te[cols])[:, 1]
p_heur = h.predict_proba(te[cols])[:, 1]
print(f'actual {y.mean():.3f} | model {p_model.mean():.3f} | heuristic {p_heur.mean():.3f}')

## The ranking survives, the level does not

In [ ]:
calibration_table(y, p_model)

In [ ]:
calibration_table(y, p_heur)

## The cause: the history feature drifts as the window lengthens

In [ ]:
pd.DataFrame({'train_median': tr[NUMERIC].median(), 'test_median': te[NUMERIC].median(),
              'train_missing': tr[NUMERIC].isna().mean().round(3),
              'test_missing': te[NUMERIC].isna().mean().round(3)})

## Error by subgroup

In [ ]:
te2 = te.assign(faixa_valor=value_band(te['valor_solicitado']))
by_group(te2, p_model, y, 'faixa_valor')

In [ ]:
by_group(te2, p_model, y, 'area')

In [ ]:
by_group(te2, p_model, y, 'UF').head(10)